In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import sys
import os


feature_path = os.path.abspath(r"C:\Users\alexg\OneDrive\Documents\NBA-Prop-Predictor")
# feature_path = os.path.abspath('/Users/alexg/Documents/Documents/Prize-Picks-Prop-Predictor')
if feature_path not in sys.path:
    sys.path.append(feature_path)

from FEATURES.features import *
from PROPS_EV.calculateEVS import *
from BACKTEST.backtest import *
from MODELS.pipeline import *

### Load Model

In [2]:
model = joblib.load('Models/xgbModel.pkl')
features = joblib.load('Models/top_features.pkl')

### Load Data

In [3]:
pd.set_option('display.max_columns', None)


s25 = pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_25.csv')
s24 = pd.read_csv('../DATA/CSV_FILES/TRAIN_DATA/PTS_TRAIN_24.csv')

df = pd.concat([s25, s24]).sort_values(by='GAME_DATE')


date = '2025-03-12'
df = df[df['GAME_DATE'] < date]

dfsData = pd.read_csv('../DATA/CSV_FILES/BACKTEST_DATA/dfs_data.csv')
dfsData = dfsData[(dfsData['BOOKMAKER'] == 'prizepicks') & (dfsData['GAME_DATE'] == date) & (dfsData['CATEGORY'] == 'player_points')]
df.tail()

C:\Users\alexg\AppData\Local\Temp\ipykernel_27840\352106883.py:13: DtypeWarning: Columns (11,12,14) have mixed types. Specify dtype option on import or set low_memory=False.
  dfsData = pd.read_csv('../DATA/CSV_FILES/BACKTEST_DATA/dfs_data.csv')


Unnamed: 0  Unnamed: 0.2     PLAYER_NAME  PLAYER_ID      MATCHUP  \
12837       12837       20826.0  Day'Ron Sharpe    1630549    BKN @ CLE   
24329       24329       20835.0    Anthony Gill    1630264    WAS @ DET   
24328       24328       20816.0       Alex Sarr    1642259    WAS @ DET   
24332       24332       20846.0   Corey Kispert    1630557    WAS @ DET   
3232         3232       20786.0     Bruce Brown    1628971  NOP vs. LAC   

      TEAM_ABBREVIATION     TEAM_ID OPP_ABBREVIATION  HOME_GAME   GAME_ID  \
12837               BKN  1610612751              CLE          0  22400940   
24329               WAS  1610612764              DET          0  22400941   
24328               WAS  1610612764              DET          0  22400941   
24332               WAS  1610612764              DET          0  22400941   
3232                NOP  1610612740              LAC          1  22400943   

        GAME_DATE WL  PTS  AST  REB  FGM  FGA  FG_PCT  FG3M  FG3A  FG3_PCT  \
12837  2025-03-11  L    8    2    7    4    6   0.667     0     0      NaN   
24329  2025-03-11  L    0    2    0    0    1   0.000     0     1    0.000   
24328  2025-03-11  L    8    2    7    2    9   0.222     1     3    0.333   
24332  2025-03-11  L   10    3    6    3    9   0.333     1     6    0.167   
3232   2025-03-11  W   12    5    5    5    9   0.556     1     2    0.500   

       FTM  FTA  FT_PCT  OREB  DREB  STL  BLK  TOV  PLUS_MINUS  FANTASY_PTS  \
12837    0    0     NaN     2     5    1    0    1          -2         21.4   
24329    0    0     NaN     0     0    0    0    0          -3          3.0   
24328    3    6     0.5     2     5    0    1    4         -22         18.4   
24332    3    3     1.0     1     5    0    0    1          -5         20.7   
3232     1    1     1.0     1     4    0    0    0          -6         25.5   

       POINT_PER_SHOT       EFG START_POSITION  COMMENT  E_OFF_RATING  \
12837           1.333  0.666667            NaN      NaN         106.1   
24329           0.000  0.000000            NaN      NaN         125.0   
24328           0.687  0.277778              C      NaN          75.1   
24332           0.969  0.388889            NaN      NaN         116.1   
3232            1.271  0.611111              G      NaN         113.8   

       E_DEF_RATING  NET_RATING  OREB_PCT  DREB_PCT  REB_PCT  AST_PCT  \
12837         108.1       -10.5     0.100     0.333    0.200    0.167   
24329         160.0       -33.3     0.000     0.000    0.000    1.000   
24328         116.8       -40.2     0.061     0.185    0.117    0.200   
24332         117.4        -8.9     0.037     0.172    0.107    0.150   
3232          125.1        -7.6     0.037     0.174    0.100    0.227   

       EFG_PCT  AST_TOV  USG_PCT  TS_PCT  E_PACE    PACE    PIE  POSS  \
12837    0.667      2.0    0.156   0.667  103.15  100.44  0.163    41   
24329    0.000      0.0    0.167   0.000   88.16  107.76  0.057     5   
24328    0.278      0.5    0.276   0.344  104.98  105.19  0.020    49   
24332    0.389      3.0    0.180   0.484  102.85  104.27  0.092    56   
3232     0.611      0.0    0.122   0.636  102.71  102.45  0.106    63   

       PACE_PER40  E_USG_PCT    MIN   SPD  DIST  ORBC  DRBC  RBC  TCHS  SAST  \
12837       83.70      0.157  19.12  4.47  1.51     5     6   11    41     0   
24329       89.80      0.200   2.45  3.88  0.18     0     0    0     5     0   
24328       87.66      0.284  22.82  4.36  1.76     3    11   14    50     0   
24332       86.89      0.188  25.78  4.65  2.11     1     5    6    44     0   
3232        85.38      0.123  29.75  4.33  2.32     5     5    9    49     0   

       FTAST  PASS  CFGM  CFGA  CFG_PCT  UFGM  UFGA  UFG_PCT  DFGM  DFGA  \
12837      0    34     4     5    0.800     0     1    0.000     2     2   
24329      0     4     0     0    0.000     0     1    0.000     0     0   
24328      0    34     1     4    0.250     1     5    0.200     8    11   
24332      0    34     1     2    0.500     2  

In [4]:
backtestData = pd.read_csv('../DATA/CSV_FILES/BACKTEST_DATA/singleBookies.csv')
backtestData = backtestData[(backtestData['ODDS'] <= 200) & (backtestData['ODDS'] >= -200)]
singleBookies = backtestData[(backtestData['CATEGORY'] == 'points') & (backtestData['GAME_DATE'] == date)]
singleBookies

,NAME,CATEGORY,SIDE,BOOKMAKER,LINE,ODDS,fair_line,fair_odds,GAME_DATE
127510,Dyson Daniels,points,over,espnbet,14.5,-125,14.5,100,2025-03-12
127511,Dyson Daniels,points,over,fanduel,14.5,-110,14.5,100,2025-03-12
127512,Dyson Daniels,points,over,draftkings,14.5,-120,14.5,100,2025-03-12
127513,Dyson Daniels,points,over,betmgm,14.5,-125,14.5,100,2025-03-12
127514,Dyson Daniels,points,over,betrivers,14.5,-112,14.5,100,2025-03-12
...,...,...,...,...,...,...,...,...,...
130910,Nikola Jokić,points,under,betmgm,27.5,-115,28.5,100,2025-03-12
130978,Zeke Nnaji,points,over,draftkings,4.5,110,4.5,120,2025-03-12
130982,Zeke Nnaji,points,under,draftkings,4.5,-140,4.5,-120,2025-03-12
130995,Naz Reid,points,over,betmgm,12.5,-105,11.5,100,2025-03-12


### Top EVs for single bets

In [5]:
results = single_bet(
    data=df,
    bookmakers=singleBookies,
    current_date=date,
    model=model,
    features=features,
    stake=5,
    simulations=10000, 
    std_window=10,
    min_std=2.0,
    max_std=10.0
)
results.sort_values(by='EV%', ascending=False).head(10)

Processing single bets...


,NAME,BOOKMAKER,CATEGORY,LINE,ODDS,SIDE,PREDICTION,RECOMMENDATION,OVER%,UNDER%,IMPLIED PROB,EV%,KELLY FULL,KELLY HALF,KELLY QUARTER,CONFIDENCE INTERVAL
843,Naji Marshall,fanduel,points,17.5,132,over,22.574394,1,0.889,0.111,0.431,106.27,0.81,0.40,0.20,"(14.5, 30.6)"
204,Aaron Wiggins,betmgm,points,2.5,105,over,14.909143,1,1.000,0.000,0.488,105.00,1.00,0.50,0.25,"(8.7, 21.0)"
859,Julian Champagnie,betmgm,points,3.5,105,over,12.385593,1,0.973,0.027,0.488,99.49,0.95,0.47,0.24,"(3.4, 21.9)"
864,Sandro Mamukelashvili,espnbet,points,4.5,100,over,9.420444,1,0.981,0.019,0.500,96.24,0.96,0.48,0.24,"(4.7, 14.1)"
863,Sandro Mamukelashvili,betmgm,points,4.5,100,over,9.420444,1,0.980,0.020,0.500,96.08,0.96,0.48,0.24,"(4.8, 14.0)"
344,Adem Bona,espnbet,points,6.5,-110,over,10.949961,0,0.987,0.013,0.524,88.33,0.97,0.49,0.24,"(7.0, 14.8)"
889,Nickeil Alexander-Walker,betmgm,points,7.5,-105,over,13.924473,1,0.964,0.036,0.512,88.11,0.93,0.46,0.23,"(7.0, 20.8)"
310,Jamal Shead,draftkings,points,15.5,-105,under,9.180629,1,0.044,0.956,0.512,86.57,0.91,0.45,0.23,"(2.2, 16.5)"
295,Quentin Grimes,fanduel,points,18.5,-104,over,24.952139,1,0.936,0.064,0.510,83.58,0.87,0.43,0.22,"(16.6, 33.5)"
313,Jamal Shead,betrivers,points,14.5,-104,under,9.180629,1,0.080,0.920,0.510,80.56,0.84,0.42,0.21,"(2.1, 16.5)"


### Top EVs for 2 leg bets

In [6]:
results = prizepickspairsEV(
    data=df,
    bookmakers=dfsData,
    current_date=date,
    model=model,
    features=features,
    stake=100,
    simulations=10000,
    std_window=10,
    min_std=2.0,
    max_std=10.0
)
results.sort_values(by='EV%', ascending=False).head(10).reset_index(drop=True)

Processing pairs...


,PLAYER 1,CATEGORY 1,BOOKMAKER 1,ODDS 1,LINE 1,SIDE 1,PREDICTION 1,MODEL_SIDE 1,OVER% 1,UNDER% 1,CONFIDENCE INTERVAL 1,PLAYER 2,CATEGORY 2,BOOKMAKER 2,ODDS 2,LINE 2,SIDE 2,PREDICTION 2,MODEL_SIDE 2,OVER% 2,UNDER% 2,CONFIDENCE INTERVAL 2,RECOMMENDED_TYPE,RECOMMENDATION,PROBABILITY,EV%,KELLY
0,Jamison Battle,player_points,prizepicks,-137,12.5,over,5.94,UNDER,0.051,0.949,"(0.7, 13.7)",Quentin Grimes,player_points,prizepicks,-137,18.5,over,24.95,OVER,0.932,0.068,"(16.5, 33.3)",UNDER/OVER,1,0.8843,1.653,0.826
1,Jamal Shead,player_points,prizepicks,-137,15.0,under,9.18,UNDER,0.059,0.941,"(2.3, 16.5)",Quentin Grimes,player_points,prizepicks,-137,18.5,over,24.95,OVER,0.932,0.068,"(16.5, 33.3)",UNDER/OVER,1,0.8765,1.630,0.815
2,Jamison Battle,player_points,prizepicks,-137,12.5,over,5.94,UNDER,0.051,0.949,"(0.7, 13.7)",Paolo Banchero,player_points,prizepicks,-137,26.5,over,17.04,UNDER,0.081,0.919,"(4.5, 30.3)",UNDER/UNDER,1,0.8726,1.618,0.809
3,Jamal Shead,player_points,prizepicks,-137,15.0,under,9.18,UNDER,0.059,0.941,"(2.3, 16.5)",Paolo Banchero,player_points,prizepicks,-137,26.5,over,17.04,UNDER,0.081,0.919,"(4.5, 30.3)",UNDER/UNDER,1,0.8650,1.595,0.797
4,Andre Drummond,player_points,prizepicks,-137,10.5,over,6.04,UNDER,0.091,0.909,"(0.9, 12.7)",Jamison Battle,player_points,prizepicks,-137,12.5,over,5.94,UNDER,0.051,0.949,"(0.7, 13.7)",UNDER/UNDER,0,0.8629,1.589,0.794
5,Jamison Battle,player_points,prizepicks,-137,12.5,over,5.94,UNDER,0.051,0.949,"(0.7, 13.7)",Kentavious Caldwell-Pope,player_points,prizepicks,-137,8.5,over,4.90,UNDER,0.094,0.906,"(0.6, 10.3)",UNDER/UNDER,0,0.8600,1.580,0.790
6,Paolo Banchero,player_points,prizepicks,-137,26.5,over,17.04,UNDER,0.081,0.919,"(4.5, 30.3)",Quentin Grimes,player_points,prizepicks,-137,18.5,over,24.95,OVER,0.932,0.068,"(16.5, 33.3)",UNDER/OVER,1,0.8566,1.570,0.785
7,Andre Drummond,player_points,prizepicks,-137,10.5,over,6.04,UNDER,0.091,0.909,"(0.9, 12.7)",Jamal Shead,player_points,prizepicks,-137,15.0,under,9.18,UNDER,0.059,0.941,"(2.3, 16.5)",UNDER/UNDER,0,0.8554,1.566,0.783
8,Jamal Shead,player_points,prizepicks,-137,15.0,under,9.18,UNDER,0.059,0.941,"(2.3, 16.5)",Kentavious Caldwell-Pope,player_points,prizepicks,-137,8.5,over,4.90,UNDER,0.094,0.906,"(0.6, 10.3)",UNDER/UNDER,0,0.8525,1.557,0.779
9,Jamison Battle,player_points,prizepicks,-137,12.5,over,5.94,UNDER,0.051,0.949,"(0.7, 13.7)",Zion Williamson,player_points,prizepicks,-137,23.0,over,26.75,OVER,0.898,0.102,"(21.1, 32.5)",UNDER/OVER,0,0.8520,1.556,0.778


In [7]:
threeLeg = prizepicks3LegEV(
    data=df,
    bookmakers=dfsData,
    current_date=date,
    model=model,
    features=features,
    stake=100,
    simulations=10000,
    std_window=10,
    min_std=2.0,
    max_std=10.0
)
threeLeg.sort_values(by='EV%', ascending=False).head(10)

Processing 3-leg parlays...


,PLAYER 1,CATEGORY 1,BOOKMAKER 1,ODDS 1,LINE 1,SIDE 1,PREDICTION 1,MODEL_SIDE 1,OVER% 1,UNDER% 1,CONFIDENCE INTERVAL 1,PLAYER 2,CATEGORY 2,BOOKMAKER 2,ODDS 2,LINE 2,SIDE 2,PREDICTION 2,MODEL_SIDE 2,OVER% 2,UNDER% 2,CONFIDENCE INTERVAL 2,PLAYER 3,CATEGORY 3,BOOKMAKER 3,ODDS 3,LINE 3,SIDE 3,PREDICTION 3,MODEL_SIDE 3,OVER% 3,UNDER% 3,CONFIDENCE INTERVAL 3,RECOMMENDED_TYPE,RECOMMENDATION,PROBABILITY,EV%,KELLY
23171,Jamal Shead,player_points,prizepicks,-137,15.0,under,9.18,UNDER,0.062,0.938,"(2.2, 16.6)",Jamison Battle,player_points,prizepicks,-137,12.5,over,5.94,UNDER,0.053,0.947,"(0.6, 13.9)",Quentin Grimes,player_points,prizepicks,-137,18.5,over,24.95,OVER,0.932,0.068,"(16.6, 33.6)",UNDER/UNDER/OVER,1,0.8272,3.963,0.793
23169,Jamal Shead,player_points,prizepicks,-137,15.0,under,9.18,UNDER,0.062,0.938,"(2.2, 16.6)",Jamison Battle,player_points,prizepicks,-137,12.5,over,5.94,UNDER,0.053,0.947,"(0.6, 13.9)",Paolo Banchero,player_points,prizepicks,-137,26.5,over,17.04,UNDER,0.081,0.919,"(4.5, 30.3)",UNDER/UNDER/UNDER,1,0.8157,3.894,0.779
24548,Jamison Battle,player_points,prizepicks,-137,12.5,over,5.94,UNDER,0.053,0.947,"(0.6, 13.9)",Paolo Banchero,player_points,prizepicks,-137,26.5,over,17.04,UNDER,0.081,0.919,"(4.5, 30.3)",Quentin Grimes,player_points,prizepicks,-137,18.5,over,24.95,OVER,0.932,0.068,"(16.6, 33.6)",UNDER/UNDER/OVER,1,0.8109,3.866,0.773
23590,Jamal Shead,player_points,prizepicks,-137,15.0,under,9.18,UNDER,0.062,0.938,"(2.2, 16.6)",Paolo Banchero,player_points,prizepicks,-137,26.5,over,17.04,UNDER,0.081,0.919,"(4.5, 30.3)",Quentin Grimes,player_points,prizepicks,-137,18.5,over,24.95,OVER,0.932,0.068,"(16.6, 33.6)",UNDER/UNDER/OVER,1,0.8033,3.820,0.764
7811,Andre Drummond,player_points,prizepicks,-137,10.5,over,6.04,UNDER,0.096,0.904,"(0.8, 12.6)",Jamal Shead,player_points,prizepicks,-137,15.0,under,9.18,UNDER,0.062,0.938,"(2.2, 16.6)",Jamison Battle,player_points,prizepicks,-137,12.5,over,5.94,UNDER,0.053,0.947,"(0.6, 13.9)",UNDER/UNDER/UNDER,0,0.8023,3.814,0.763
23178,Jamal Shead,player_points,prizepicks,-137,15.0,under,9.18,UNDER,0.062,0.938,"(2.2, 16.6)",Jamison Battle,player_points,prizepicks,-137,12.5,over,5.94,UNDER,0.053,0.947,"(0.6, 13.9)",Zion Williamson,player_points,prizepicks,-137,23.0,over,26.75,OVER,0.903,0.097,"(21.2, 32.4)",UNDER/UNDER/OVER,0,0.8012,3.807,0.761
7897,Andre Drummond,player_points,prizepicks,-137,10.5,over,6.04,UNDER,0.096,0.904,"(0.8, 12.6)",Jamison Battle,player_points,prizepicks,-137,12.5,over,5.94,UNDER,0.053,0.947,"(0.6, 13.9)",Quentin Grimes,player_points,prizepicks,-137,18.5,over,24.95,OVER,0.932,0.068,"(16.6, 33.6)",UNDER/UNDER/OVER,0,0.7976,3.786,0.757
24573,Jamison Battle,player_points,prizepicks,-137,12.5,over,5.94,UNDER,0.053,0.947,"(0.6, 13.9)",Quentin Grimes,player_points,prizepicks,-137,18.5,over,24.95,OVER,0.932,0.068,"(16.6, 33.6)",Zion Williamson,player_points,prizepicks,-137,23.0,over,26.75,OVER,0.903,0.097,"(21.2, 32.4)",UNDER/OVER/OVER,0,0.7965,3.779,0.756
23158,Jamal Shead,player_points,prizepicks,-137,15.0,under,9.18,UNDER,0.062,0.938,"(2.2, 16.6)",Jamison Battle,player_points,prizepicks,-137,12.5,over,5.94,UNDER,0.053,0.947,"(0.6, 13.9)",Kentavious Caldwell-Pope,player_points,prizepicks,-137,8.5,over,4.90,UNDER,0.103,0.897,"(0.7, 10.4)",UNDER/UNDER/UNDER,0,0.7960,3.776,0.755
24383,Jamison Battle,player_points,prizepicks,-137,12.5,over,5.94,UNDER,0.053,0.947,"(0.6, 13.9)",Kentavious Caldwell-Pope,player_points,prizepicks,-137,8.5,over,4.90,UNDER,0.103,0.897,"(0.7, 10.4)",Quentin Grimes,player_points,prizepicks,-137,18.5,over,24.95,OVER,0.932,0.068,"(16.6, 33.6)",UNDER/UNDER/OVER,0,0.7914,3.748,0.750
